# Federated Learning — Non-IID Data Distribution Experiment


| Scenario | Description |
|---|---|
| **IID** | Original paper — balanced uniform distribution across 4 clients |
| **Non-IID Mild (α=0.5)** | Dirichlet — each client dominated by a few classes |
| **Non-IID Severe (α=0.1)** | Dirichlet — extreme skew, 1–2 dominant classes per client |

**Data split (proper three-way).** `train/` is split (stratified) into a balanced **validation** set and a **client pool**; each scenario (IID / Non-IID Mild / Severe) partitions the *pool* across clients. The global model is monitored on the **validation** set each round (per-round curves), and the final per-class comparison is reported on the **held-out test set** (`valid/`), which is never used during training.

## 1. Imports & Memory Configuration

In [ ]:
import gc
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from collections import Counter
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, classification_report
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dropout, Dense
from tensorflow.keras.utils import to_categorical, normalize
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall

import warnings
warnings.filterwarnings('ignore')

# ── Memory-saving settings ─────────────────────────────────────────────────
# Set to (128,128) if you have >16 GB RAM available
IMG_SIZE     = (64, 64)   # ← KEY: smaller = much less RAM
NUM_CLASSES  = 10
NUM_CLIENTS  = 4
NUM_ROUNDS   = 5          # same as original paper
LOCAL_EPOCHS = 3          # same as original paper
BATCH_SIZE   = 16
RANDOM_SEED  = 123

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

CLASS_NAMES = [
    'Bacterial Spot', 'Early Blight', 'Late Blight', 'Leaf Mold',
    'Septoria Leaf Spot', 'Spider Mites', 'Target Spot',
    'Yellow Leaf Curl Virus', 'Mosaic Virus', 'Healthy'
]

print(f"Image size : {IMG_SIZE}")
print(f"TF version : {tf.__version__}")
print("Ready.")

## 2. Memory-Efficient Data Loading

Key savings vs original notebook:
- Uses `float16` instead of `float32` → **50% less RAM**
- Loads images directly without intermediate `dtype=object` array
- Clears intermediate variables immediately with `del` + `gc.collect()`

In [ ]:
tomato_train = './new-plant-diseases-dataset/train/'
tomato_valid = './new-plant-diseases-dataset/valid/'

def load_data_efficient(dir_path, img_size=IMG_SIZE):
    """
    Memory-efficient loader:
    - Builds lists incrementally, avoids dtype=object
    - Returns float16 arrays to halve memory usage
    """
    X_list, Y_list = [], []
    i = 0
    labels = {}
    dirs = sorted([
        p for p in os.listdir(dir_path)
        if not p.startswith('.') and 'Tomato__' in p
    ])
    for path in tqdm(dirs, desc=f'Loading {os.path.basename(dir_path)}'):
        labels[i] = path
        sub_dir = os.path.join(dir_path, path)
        for file in os.listdir(sub_dir):
            if file.startswith('.'):
                continue
            file_path = os.path.join(sub_dir, file)
            if not os.path.isfile(file_path):
                continue
            img = cv2.imread(file_path)
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, img_size)         # resize here, not after
            X_list.append(img)
            Y_list.append(i)
        i += 1

    # Build final arrays as float16 to save RAM
    X = np.array(X_list, dtype=np.float16)
    Y = np.array(Y_list, dtype=np.int8)
    del X_list, Y_list
    gc.collect()
    print(f"  Loaded {len(X)} images | RAM ~{X.nbytes / 1e6:.1f} MB")
    return X, Y, labels


print("Loading training data...")
X_train_raw, y_train_raw, labels = load_data_efficient(tomato_train)

print("\nLoading validation data...")
X_valid_raw, y_valid_raw, _ = load_data_efficient(tomato_valid)

print(f"\nTrain: {X_train_raw.shape} | dtype: {X_train_raw.dtype}")
print(f"Valid: {X_valid_raw.shape} | dtype: {X_valid_raw.dtype}")

In [ ]:
# ── Preprocess + proper 3-way split ───────────────────────────────────────────
# train/ -> client POOL + stratified VALIDATION ; valid/ -> held-out TEST set.
X_train_raw = normalize(X_train_raw.astype(np.float32), axis=1)
X_valid_raw = normalize(X_valid_raw.astype(np.float32), axis=1)

# Held-out test set (from valid/).
X_test, y_test_int = shuffle(X_valid_raw, y_valid_raw, random_state=RANDOM_SEED)
y_test_cat = to_categorical(y_test_int, num_classes=NUM_CLASSES)

# Carve a stratified validation set out of train/; the rest is the client pool.
X_pool, X_val, y_pool, y_val = train_test_split(
    X_train_raw, y_train_raw, test_size=0.2, stratify=y_train_raw,
    random_state=RANDOM_SEED)
y_val_cat = to_categorical(y_val, num_classes=NUM_CLASSES)

# The partitioning functions consume the client pool as (X_train, y_train).
X_train, y_train = shuffle(X_pool, y_pool, random_state=RANDOM_SEED)

del X_train_raw, X_valid_raw, y_train_raw, y_valid_raw, X_pool, y_pool
gc.collect()

print(f"Client pool : {X_train.shape} | {X_train.nbytes/1e6:.1f} MB")
print(f"Validation  : {X_val.shape}")
print(f"Test (held-out): {X_test.shape}")
print(f"Pool class distribution: {dict(sorted(Counter(y_train).items()))}")


## 3. Model Definition — SimpleNetAugDR-3
*(Identical architecture to the paper — adapted input size)*

In [ ]:
def create_model():
    model = Sequential([
        Conv2D(64, (4, 4), activation='relu',
               input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (4, 4), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), activation='relu'),   # 64/64/64 -> matches Figure 2 & benchmark NB01
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax'),
    ])
    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(learning_rate=0.001),
        metrics=[
            'accuracy',
            Precision(name='precision'),
            Recall(name='recall'),
            'AUC',
            'top_k_categorical_accuracy',
        ]
    )
    return model

# Quick architecture check
tmp = create_model()
tmp.summary()
del tmp
gc.collect()

## 4. Data Partitioning Functions

In [ ]:
def iid_split(X, y, num_clients):
    """
    IID: shuffle then split equally.
    Returns list of (X_client, y_client_categorical).
    """
    X_s, y_s = shuffle(X, y, random_state=RANDOM_SEED)
    shard = len(X_s) // num_clients
    client_data = []
    for i in range(num_clients):
        Xc = X_s[i*shard:(i+1)*shard].copy()
        yc = to_categorical(y_s[i*shard:(i+1)*shard], NUM_CLASSES)
        client_data.append((Xc, yc))
    return client_data


def dirichlet_split(X, y, num_clients, alpha, seed=RANDOM_SEED):
    """
    Non-IID via Dirichlet distribution.
    alpha=0.5 → mild skew | alpha=0.1 → severe skew.
    """
    rng = np.random.default_rng(seed)
    class_indices = {c: np.where(y == c)[0].copy() for c in range(NUM_CLASSES)}
    client_indices = [[] for _ in range(num_clients)]

    for c in range(NUM_CLASSES):
        idx = class_indices[c]
        rng.shuffle(idx)
        props = rng.dirichlet(np.repeat(alpha, num_clients))
        splits = (np.cumsum(props) * len(idx)).astype(int)[:-1]
        for cid, chunk in enumerate(np.split(idx, splits)):
            client_indices[cid].extend(chunk.tolist())

    client_data = []
    for cid in range(num_clients):
        idx = np.array(client_indices[cid])
        rng.shuffle(idx)
        Xc = X[idx].copy()
        yc = to_categorical(y[idx], NUM_CLASSES)
        client_data.append((Xc, yc))
    return client_data


def print_distribution(client_data, scenario_name):
    print(f"\n  {scenario_name}")
    for i, (Xc, yc) in enumerate(client_data):
        counts = dict(sorted(Counter(np.argmax(yc, axis=1)).items()))
        print(f"    Client {i+1}: {len(Xc):5d} samples | {counts}")

print("Partitioning functions defined.")

## 5. Visualise Class Distributions (build partitions once for the plot)

In [ ]:
# Build all three partitions just for the distribution plot
partitions_for_plot = {
    'IID (α→∞)':           iid_split(X_train, y_train, NUM_CLIENTS),
    'Non-IID Mild (α=0.5)':  dirichlet_split(X_train, y_train, NUM_CLIENTS, alpha=0.5),
    'Non-IID Severe (α=0.1)':dirichlet_split(X_train, y_train, NUM_CLIENTS, alpha=0.1),
}

for name, cdata in partitions_for_plot.items():
    print_distribution(cdata, name)

# Plot
COLORS_CLIENT = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
fig, axes = plt.subplots(3, NUM_CLIENTS, figsize=(16, 10), sharey=False)
fig.suptitle('Class Distribution per Client — IID vs Non-IID Scenarios',
             fontsize=13, fontweight='bold', y=1.01)

for row_idx, (scenario_name, cdata) in enumerate(partitions_for_plot.items()):
    for col_idx, (Xc, yc) in enumerate(cdata):
        ax = axes[row_idx][col_idx]
        counts = np.array([np.sum(np.argmax(yc, axis=1) == c)
                           for c in range(NUM_CLASSES)])
        ax.bar(range(NUM_CLASSES), counts,
               color=COLORS_CLIENT[col_idx], alpha=0.8, edgecolor='white')
        ax.set_xticks(range(NUM_CLASSES))
        ax.set_xticklabels([f'C{i}' for i in range(NUM_CLASSES)], fontsize=7)
        ax.set_title(f'Client {col_idx+1}', fontsize=10, fontweight='bold')
        ax.grid(True, axis='y', alpha=0.3)
        if col_idx == 0:
            ax.set_ylabel(f'{scenario_name}\n\nCount', fontsize=8)
        ax.text(0.97, 0.96, f'n={len(Xc)}', transform=ax.transAxes,
                ha='right', va='top', fontsize=7,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig('fig_noniid_class_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_noniid_class_distributions.png")

# Free partitions — they will be rebuilt one at a time during training
del partitions_for_plot
gc.collect()
print("Memory freed.")

## 6. FL Training — One Scenario at a Time

**Memory strategy:** each scenario is built, trained, evaluated, then **immediately deleted** before the next one starts. Only the results (numbers) are kept between scenarios.

In [ ]:
def fedavg(local_weights, sizes):
    """Weighted FedAvg: sum_k (n_k / N) * w_k."""
    total = float(sum(sizes))
    return [sum((sizes[k] / total) * lw[layer] for k, lw in enumerate(local_weights))
            for layer in range(len(local_weights[0]))]


def run_fl_scenario(scenario_name, client_data, num_rounds=NUM_ROUNDS):
    """Run FedAvg for one data-distribution scenario.

    Per-round metrics are computed on the VALIDATION set (the held-out test set is
    reserved for the final per-class comparison). Local models are freed each round
    to keep RAM low. Returns (per-round DataFrame, final global model).
    """
    print(f"\n{'='*65}\n  SCENARIO: {scenario_name}\n{'='*65}")
    global_model = create_model()
    sizes = [len(Xc) for Xc, _ in client_data]
    results = {'round': [], 'loss': [], 'accuracy': [],
               'precision': [], 'recall': [], 'f1_score': []}

    for round_num in range(1, num_rounds + 1):
        global_w = global_model.get_weights()
        local_weights = []
        for cid in range(NUM_CLIENTS):
            local_model = create_model()
            local_model.set_weights(global_w)
            Xc, yc = client_data[cid]
            local_model.fit(Xc, yc, batch_size=BATCH_SIZE, epochs=LOCAL_EPOCHS, verbose=0)
            local_weights.append(local_model.get_weights())
            del local_model; gc.collect()

        global_model.set_weights(fedavg(local_weights, sizes))   # weighted FedAvg
        del local_weights; gc.collect()

        # Evaluate the global model on the VALIDATION set.
        y_pred = global_model.predict(X_val, verbose=0, batch_size=64).argmax(axis=1)
        y_true = y_val_cat.argmax(axis=1)
        loss = global_model.evaluate(X_val, y_val_cat, verbose=0)[0]
        results['round'].append(round_num)
        results['loss'].append(round(loss, 4))
        results['accuracy'].append(round((y_pred == y_true).mean() * 100, 2))
        results['precision'].append(round(precision_score(y_true, y_pred, average='weighted', zero_division=0) * 100, 2))
        results['recall'].append(round(recall_score(y_true, y_pred, average='weighted', zero_division=0) * 100, 2))
        results['f1_score'].append(round(f1_score(y_true, y_pred, average='weighted', zero_division=0) * 100, 2))
        print(f"  Round {round_num}/{num_rounds} | val acc {results['accuracy'][-1]:.2f}% | "
              f"F1 {results['f1_score'][-1]:.2f}")

    return pd.DataFrame(results), global_model


### 6a. Run IID Scenario

In [ ]:
client_data_iid = iid_split(X_train, y_train, NUM_CLIENTS)

df_iid, model_iid = run_fl_scenario('IID (α→∞)', client_data_iid)

# Free client data immediately
del client_data_iid
gc.collect()

df_iid.to_csv('fl_noniid_IID.csv', index=False)
print("\nIID results saved.")
print(df_iid.to_string(index=False))

### 6b. Run Non-IID Mild Scenario

In [ ]:
client_data_mild = dirichlet_split(X_train, y_train, NUM_CLIENTS, alpha=0.5)

df_mild, model_mild = run_fl_scenario('Non-IID Mild (α=0.5)', client_data_mild)

del client_data_mild
gc.collect()

df_mild.to_csv('fl_noniid_Mild.csv', index=False)
print("\nNon-IID Mild results saved.")
print(df_mild.to_string(index=False))

### 6c. Run Non-IID Severe Scenario

In [ ]:
client_data_severe = dirichlet_split(X_train, y_train, NUM_CLIENTS, alpha=0.1)

df_severe, model_severe = run_fl_scenario('Non-IID Severe (α=0.1)', client_data_severe)

del client_data_severe
gc.collect()

df_severe.to_csv('fl_noniid_Severe.csv', index=False)
print("\nNon-IID Severe results saved.")
print(df_severe.to_string(index=False))

## 7. Summary table & statistical analysis (validation set)

In [ ]:
all_results = {
    'IID (α→∞)':            df_iid,
    'Non-IID Mild (α=0.5)':  df_mild,
    'Non-IID Severe (α=0.1)':df_severe,
}

print("=" * 70)
print("FINAL ROUND COMPARISON SUMMARY (validation set)")
print("=" * 70)

summary_rows = []
iid_acc = None
for name, df in all_results.items():
    last = df[df['round'] == NUM_ROUNDS].iloc[0]
    acc  = last['accuracy']
    if iid_acc is None:
        iid_acc = acc
    drop = acc - iid_acc
    summary_rows.append({
        'Scenario':      name,
        'Accuracy (%)':  f"{acc:.2f}",
        'Precision (%)': f"{last['precision']:.2f}",
        'Recall (%)':    f"{last['recall']:.2f}",
        'F1-Score (%)':  f"{last['f1_score']:.2f}",
        'Δ vs IID (pp)': f"{drop:+.2f}",
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv('fl_noniid_summary.csv', index=False)
print("\nSaved: fl_noniid_summary.csv")

print("\n" + "=" * 70)
print("CONVERGENCE STABILITY ANALYSIS")
print("=" * 70)
for name, df in all_results.items():
    acc   = df['accuracy'].values
    delta = np.abs(np.diff(acc))
    print(f"\n  {name}")
    print(f"    Max |Δ accuracy| : {delta.max():.4f} pp")
    print(f"    Avg |Δ accuracy| : {delta.mean():.4f} pp")
    print(f"    Std of accuracy  : {acc.std():.4f} pp")
    print(f"    Final accuracy   : {acc[-1]:.2f}%")

## 8. Publication-Quality Figures

In [ ]:
SCENARIO_COLORS = {
    'IID (α→∞)':            '#2196F3',
    'Non-IID Mild (α=0.5)':  '#FF9800',
    'Non-IID Severe (α=0.1)':'#F44336',
}
SCENARIO_MARKERS = {
    'IID (α→∞)':            'o-',
    'Non-IID Mild (α=0.5)':  's--',
    'Non-IID Severe (α=0.1)':'D-.',
}
rounds = list(range(1, NUM_ROUNDS + 1))

plt.rcParams.update({
    'font.size': 12, 'axes.titlesize': 13,
    'axes.labelsize': 12, 'legend.fontsize': 10,
})

# ── Figure 1: Accuracy & Loss ──────────────────────────────────────────────
fig1, axes = plt.subplots(1, 2, figsize=(14, 5))
fig1.suptitle('FL Convergence Stability — IID vs Non-IID (5 Rounds)',
              fontsize=13, fontweight='bold', y=1.02)

for name, df in all_results.items():
    axes[0].plot(df['round'], df['accuracy'],
                 SCENARIO_MARKERS[name], color=SCENARIO_COLORS[name],
                 linewidth=2, markersize=7, label=name)
    axes[1].plot(df['round'], df['loss'],
                 SCENARIO_MARKERS[name], color=SCENARIO_COLORS[name],
                 linewidth=2, markersize=7, label=name)

for ax, ylabel, title in zip(
    axes,
    ['Global Accuracy (%)', 'Global Loss'],
    ['Accuracy per FL Round', 'Loss per FL Round']
):
    ax.set_xlabel('FL Round')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.set_xticks(rounds)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_noniid_accuracy_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_noniid_accuracy_loss.png")

In [ ]:
# ── Figure 2: Precision / Recall / F1 ─────────────────────────────────────
fig2, axes = plt.subplots(1, 3, figsize=(18, 5))
fig2.suptitle('Precision, Recall & F1-Score — IID vs Non-IID (5 Rounds)',
              fontsize=13, fontweight='bold', y=1.02)

for ax, (col, label) in zip(axes, [('precision','Precision (%)'),
                                     ('recall','Recall (%)'),
                                     ('f1_score','F1-Score (%)')]):
    for name, df in all_results.items():
        ax.plot(df['round'], df[col],
                SCENARIO_MARKERS[name], color=SCENARIO_COLORS[name],
                linewidth=2, markersize=7, label=name)
    ax.set_xlabel('FL Round')
    ax.set_ylabel(label)
    ax.set_title(label.split(' (')[0], fontweight='bold')
    ax.set_xticks(rounds)
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_noniid_precision_recall_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_noniid_precision_recall_f1.png")

In [ ]:
# ── Figure 3: Final-round grouped bar chart ────────────────────────────────
final_metrics = {}
for name, df in all_results.items():
    last = df[df['round'] == NUM_ROUNDS].iloc[0]
    final_metrics[name] = {
        'Accuracy':  last['accuracy'],
        'Precision': last['precision'],
        'Recall':    last['recall'],
        'F1-Score':  last['f1_score'],
    }

metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x     = np.arange(len(metric_names))
width = 0.25

fig3, ax = plt.subplots(figsize=(12, 6))
for i, (name, metrics) in enumerate(final_metrics.items()):
    vals   = [metrics[m] for m in metric_names]
    offset = (i - 1) * width
    bars   = ax.bar(x + offset, vals, width, label=name,
                    color=SCENARIO_COLORS[name], alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.2,
                f'{v:.2f}', ha='center', va='bottom', fontsize=8, rotation=45)

ax.set_xlabel('Metric', fontweight='bold')
ax.set_ylabel('Score (%)', fontweight='bold')
ax.set_title(f'Final Round ({NUM_ROUNDS}) Performance — IID vs Non-IID',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylim(max(0, min([v for m in final_metrics.values()
                         for v in m.values()]) - 5), 105)
ax.legend(loc='lower right')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig_noniid_final_round_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_noniid_final_round_comparison.png")

In [ ]:
# ── Figure 4: Round-to-round stability ────────────────────────────────────
fig4, ax = plt.subplots(figsize=(10, 5))
for name, df in all_results.items():
    delta = np.abs(np.diff(df['accuracy'].values))
    ax.plot(range(2, NUM_ROUNDS + 1), delta,
            SCENARIO_MARKERS[name], color=SCENARIO_COLORS[name],
            linewidth=2, markersize=7, label=name)

ax.axhline(y=0.5, color='black', linestyle='--', linewidth=1.5,
           label='Stability threshold (0.5 pp)')
ax.set_xlabel('FL Round')
ax.set_ylabel('|ΔAccuracy| (pp)')
ax.set_title('Round-to-Round Accuracy Fluctuation — Convergence Stability',
             fontsize=13, fontweight='bold')
ax.set_xticks(range(2, NUM_ROUNDS + 1))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_noniid_convergence_stability.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_noniid_convergence_stability.png")

## 9. Held-out test evaluation — final per-class F1 (Figure 15)

In [ ]:
all_models = {
    'IID (α→∞)':            model_iid,
    'Non-IID Mild (α=0.5)':  model_mild,
    'Non-IID Severe (α=0.1)':model_severe,
}

# Final comparison on the HELD-OUT TEST set (valid/), reported once per scenario.
test_rows = []
for name, model in all_models.items():
    yp = model.predict(X_test, verbose=0, batch_size=64).argmax(axis=1)
    yt = y_test_cat.argmax(axis=1)
    test_rows.append({
        'Scenario': name,
        'Accuracy (%)':  round((yp == yt).mean() * 100, 2),
        'Precision (%)': round(precision_score(yt, yp, average='weighted', zero_division=0) * 100, 2),
        'Recall (%)':    round(recall_score(yt, yp, average='weighted', zero_division=0) * 100, 2),
        'F1-Score (%)':  round(f1_score(yt, yp, average='weighted', zero_division=0) * 100, 2),
    })
test_summary_df = pd.DataFrame(test_rows)
print("Held-out TEST set — final scenario comparison:")
print(test_summary_df.to_string(index=False))
test_summary_df.to_csv('fl_noniid_test_summary.csv', index=False)
print()


y_true = y_test_cat.argmax(axis=1)
per_class_f1 = {}

for name, model in all_models.items():
    y_pred = model.predict(X_test, verbose=0, batch_size=64).argmax(axis=1)
    print(f"\n  {name}")
    report = classification_report(
        y_true, y_pred,
        target_names=[f'C{i}: {CLASS_NAMES[i][:12]}' for i in range(NUM_CLASSES)],
        digits=4, output_dict=True
    )
    print(classification_report(
        y_true, y_pred,
        target_names=[f'C{i}: {CLASS_NAMES[i][:12]}' for i in range(NUM_CLASSES)],
        digits=4
    ))
    per_class_f1[name] = [
        report[f'C{i}: {CLASS_NAMES[i][:12]}']['f1-score'] * 100
        for i in range(NUM_CLASSES)
    ]

# Heatmap
heatmap_df = pd.DataFrame(
    per_class_f1,
    index=[f'C{i}: {CLASS_NAMES[i][:14]}' for i in range(NUM_CLASSES)]
).T

fig5, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    heatmap_df, ax=ax, annot=True, fmt='.1f',
    cmap='RdYlGn', vmin=50, vmax=100,
    linewidths=0.5, annot_kws={'size': 9},
    cbar_kws={'label': 'F1-Score (%)'}
)
ax.set_title('Per-Class F1-Score — Final Global Model (IID vs Non-IID)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Disease Class', fontsize=11)
ax.set_ylabel('Scenario', fontsize=11)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('fig_noniid_perclass_f1_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_noniid_perclass_f1_heatmap.png")